In [0]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo destino")
catalogo = dbutils.widgets.get("catalogo")
print(f"Ejecutando COPY INTO en catálogo: {catalogo}")

In [0]:
spark.sql(f"USE CATALOG {catalogo}")

# 1. Bronze Catálogo (JSON arreglo multilínea)
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.catalogo (
  producto_id STRING,
  nombre_producto STRING,
  categoria STRING,
  marca STRING,
  precio_lista STRING,
  activo STRING,
  _ingested_at TIMESTAMP,
  _source_file STRING
)
""")

spark.sql(f"""
COPY INTO bronze.catalogo
FROM (
  SELECT
    producto_id::STRING,
    nombre_producto::STRING,
    categoria::STRING,
    marca::STRING,
    precio_lista::STRING,
    activo::STRING,
    current_timestamp() AS _ingested_at,
    _metadata.file_name AS _source_file
  FROM '/Volumes/{catalogo}/default/vol_landing/catalogo/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true', 'primitivesAsString' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false')
""")

# 2. Bronze Sucursales (CSV único)
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.sucursales (
  sucursal_id STRING,
  nombre_sucursal STRING,
  distrito STRING,
  ciudad STRING,
  region STRING,
  fecha_apertura STRING,
  _ingested_at TIMESTAMP,
  _source_file STRING
)
""")

spark.sql(f"""
COPY INTO bronze.sucursales
FROM (
  SELECT
    sucursal_id::STRING,
    nombre_sucursal::STRING,
    distrito::STRING,
    ciudad::STRING,
    region::STRING,
    fecha_apertura::STRING,
    current_timestamp() AS _ingested_at,
    _metadata.file_name AS _source_file
  FROM '/Volumes/{catalogo}/default/vol_landing/sucursales/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'false')
""")

print("COPY INTO completado con éxito para Catálogo y Sucursales.")